# Data Inspection and Feature Engineering

## Objective

This notebook turns the original relational European soccer data into one table that can be used for match-outcome modelling.

It uses the seven extracted CSV tables in `data/extracted/`. We first learn what one row means in each table and how the tables connect. We then attach only historical team and player information, calculate recent form, and create home-minus-away comparison features.

The main output is `data/processed/model_data_engineered.csv`. The existing file is treated as a protected project artifact: a fresh file is written only when it does not already exist. Notebook 02 explores the source tables, and Notebook 03 prepares the engineered table for modelling.

**Prediction target:** Home Win, Draw, or Away Win.

**Leakage rule:** no record dated after a match may be used to describe that match. Because the source provides dates rather than times, the preserved historical joins use the latest snapshot dated *on or before* the match date; this limitation is documented again where the joins occur.

## 1. Imports and Project Paths

`Path` keeps file locations portable. The small check below supports starting Jupyter from either the repository root or the `notebooks` folder, so the notebook does not depend on one person's absolute file path.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "data/extracted").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate a project root containing data/extracted."
    )


PROJECT_ROOT = find_project_root()

EXTRACTED_DIR = PROJECT_ROOT / "data" / "extracted"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load the Relational Tables

The database is relational: information is stored once and connected through IDs.

| Table | What one row means | Role |
|---|---|---|
| `Country` | one country | reference/lookup |
| `League` | one league linked to a country | reference/lookup |
| `Match` | one played fixture | transactional history and modelling base |
| `Team` | one team | reference/lookup |
| `Team_Attributes` | one dated tactical snapshot for a team | time-varying history |
| `Player` | one player | reference/lookup |
| `Player_Attributes` | one dated ability snapshot for a player | time-varying history |

The important relationships are:

```text
Country  1 ─── many  League  1 ─── many  Match
Team     1 ─── many  Match (joined once as home and once as away)
Team     1 ─── many  Team_Attributes
Player   1 ─── many  Player_Attributes
Match stores up to 11 home-player IDs and 11 away-player IDs
```

In [2]:
match = pd.read_csv(EXTRACTED_DIR / "Match.csv")
team = pd.read_csv(EXTRACTED_DIR / "Team.csv")
team_attributes = pd.read_csv(EXTRACTED_DIR / "Team_Attributes.csv")
player = pd.read_csv(EXTRACTED_DIR / "Player.csv")
player_attributes = pd.read_csv(EXTRACTED_DIR / "Player_Attributes.csv")
league = pd.read_csv(EXTRACTED_DIR / "League.csv")
country = pd.read_csv(EXTRACTED_DIR / "Country.csv")

raw_match_count = len(match)

table_inventory = pd.DataFrame(
    {
        "Table": [
            "Country", "League", "Match", "Team",
            "Team_Attributes", "Player", "Player_Attributes",
        ],
        "Rows": [
            len(country), len(league), len(match), len(team),
            len(team_attributes), len(player), len(player_attributes),
        ],
        "Columns": [
            country.shape[1], league.shape[1], match.shape[1], team.shape[1],
            team_attributes.shape[1], player.shape[1], player_attributes.shape[1],
        ],
    }
)
table_inventory

,Table,Rows,Columns
0,Country,11,2
1,League,11,3
2,Match,25979,115
3,Team,299,5
4,Team_Attributes,1458,25
5,Player,11060,7
6,Player_Attributes,183978,42


### Finding

`Match` is the central table because every final modelling row must represent one match. The two attribute tables contain repeated team or player IDs by design: those repetitions are historical snapshots, not duplicate entities.

## 3. Inspect and Reduce the Source Tables

### 3.1 Match table

Betting odds are not part of the chosen methodology. XML event fields such as shots, cards, and possession describe what happened during a match, so using them would leak post-match information. We remove both groups here.

The final score columns are temporarily retained only to create the target. They will later be excluded from candidate predictors.

In [3]:
# create a copy of the original Match table
match_selected = match.copy()

print("Original shape:", match.shape)
print("Copied shape:", match_selected.shape)

betting_columns = [
    "B365H", "B365D", "B365A",
    "BWH", "BWD", "BWA",
    "IWH", "IWD", "IWA",
    "LBH", "LBD", "LBA",
    "PSH", "PSD", "PSA",
    "WHH", "WHD", "WHA",
    "SJH", "SJD", "SJA",
    "VCH", "VCD", "VCA",
    "GBH", "GBD", "GBA",
    "BSH", "BSD", "BSA"
]
match_selected.drop(
    columns=betting_columns,
    errors="ignore",
    inplace=True
)
print("After removing betting columns:")
print(match_selected.shape)

post_match_columns = [
    "goal",
    "shoton",
    "shotoff",
    "foulcommit",
    "card",
    "cross",
    "corner",
    "possession"
]
match_selected.drop(
    columns=post_match_columns,
    errors="ignore",
    inplace=True
)
print("Current Match table shape:", match_selected.shape)
match_selected.columns.tolist()

display(match_selected[["id", "season", "date", "home_team_goal", "away_team_goal"]].head())

Original shape: (25979, 115)
Copied shape: (25979, 115)
After removing betting columns:
(25979, 85)
Current Match table shape: (25979, 77)


,id,season,date,home_team_goal,away_team_goal
0,1,2008/2009,2008-08-17 00:00:00,1,1
1,2,2008/2009,2008-08-16 00:00:00,0,0
2,3,2008/2009,2008-08-16 00:00:00,0,3
3,4,2008/2009,2008-08-17 00:00:00,5,0
4,5,2008/2009,2008-08-16 00:00:00,1,3


### 3.2 Reference and time-varying tables

For lookup tables, IDs are the reliable join keys; names are descriptive labels. For historical tables, both the entity ID and record date matter. Only the tactical and player-skill fields used by the existing engineering workflow are retained below.

In [4]:
team_selected = team[
    [
        "team_api_id",
        "team_long_name"
    ]
].copy()

team_selected.head(10)

team_attributes_selected = team_attributes[
    [
        # join + time
        "team_api_id",
        "date",

        # build-up play
        "buildUpPlaySpeed",
        "buildUpPlayPassing",
        "buildUpPlayDribblingClass",
        "buildUpPlayPositioningClass",

        # chance creation
        "chanceCreationPassing",
        "chanceCreationCrossing",
        "chanceCreationShooting",
        "chanceCreationPositioningClass",

        # defence
        "defencePressure",
        "defenceAggression",
        "defenceTeamWidth",
        "defenceDefenderLineClass"
    ]
].copy()

team_attributes_selected["date"] = pd.to_datetime(
    team_attributes_selected["date"]
)
team_attributes_selected.head()

player_selected = player[
    [
        "player_api_id",
        "player_name",
        "birthday",
        "height",
        "weight"
    ]
].copy()
player_selected["birthday"] = pd.to_datetime(
    player_selected["birthday"]
)

player_attributes_selected = player_attributes.drop(
    columns=[
        "id",
        "player_fifa_api_id"
    ]
).copy()
player_attributes_selected["date"] = pd.to_datetime(
    player_attributes_selected["date"]
)
print(player_attributes_selected.shape)
player_attributes_selected.head()

reference_summary = pd.DataFrame(
    {
        "Table": ["Country", "League", "Team", "Player"],
        "Unique key values": [
            country["id"].nunique(),
            league["id"].nunique(),
            team["team_api_id"].nunique(),
            player["player_api_id"].nunique(),
        ],
        "Rows": [len(country), len(league), len(team), len(player)],
    }
)
display(reference_summary)
print("Selected Team_Attributes shape:", team_attributes_selected.shape)
print("Selected Player_Attributes shape:", player_attributes_selected.shape)

(183978, 40)


,Table,Unique key values,Rows
0,Country,11,11
1,League,11,11
2,Team,299,299
3,Player,11060,11060


Selected Team_Attributes shape: (1458, 14)
Selected Player_Attributes shape: (183978, 40)


### Finding

`Team_Attributes` and `Player_Attributes` are time-varying. A model should not treat an old and a new rating as interchangeable, and it must never select a snapshot dated after the match. The reduced `Player` table is useful for understanding the source and validating IDs; height, weight, birthday, and player name are not used in the current engineered model features.

## 4. Validate Relationships and Build the Match Base

Before joining tables, we check whether foreign keys in the match/history data exist in their reference tables. A rate of `1.0` means complete key coverage.

In [5]:
home_team_matches = match_selected[
    "home_team_api_id"
].isin(team["team_api_id"]).mean()

away_team_matches = match_selected[
    "away_team_api_id"
].isin(team["team_api_id"]).mean()

print("Home team IDs found in Team:", home_team_matches)
print("Away team IDs found in Team:", away_team_matches)

team_attribute_matches = team_attributes_selected[
    "team_api_id"
].isin(team["team_api_id"]).mean()

print(
    "Team Attribute IDs found in Team:",
    team_attribute_matches
)

home_player_columns = [
    f"home_player_{i}"
    for i in range(1, 12)
]

away_player_columns = [
    f"away_player_{i}"
    for i in range(1, 12)
]

home_players = (
    match_selected[home_player_columns]
    .stack()
    .dropna()
)

home_player_match_rate = home_players.isin(
    player["player_api_id"]
).mean()

print(
    "Home player IDs found in Player:",
    home_player_match_rate
)

away_players = (
    match_selected[away_player_columns]
    .stack()
    .dropna()
)

away_player_match_rate = away_players.isin(
    player["player_api_id"]
).mean()

print(
    "Away player IDs found in Player:",
    away_player_match_rate
)

player_attribute_match_rate = (
    player_attributes_selected["player_api_id"]
    .isin(player["player_api_id"])
    .mean()
)

print(
    "Player Attribute IDs found in Player:",
    player_attribute_match_rate
)

Home team IDs found in Team: 1.0
Away team IDs found in Team: 1.0
Team Attribute IDs found in Team: 1.0
Home player IDs found in Player: 1.0
Away player IDs found in Player: 1.0
Player Attribute IDs found in Player: 1.0


### 4.1 Create the prediction target

The target is determined from the final score: more home goals gives `Home Win`, fewer gives `Away Win`, and equal scores give `Draw`. Scores are valid for creating the label but not for predicting it.

We also attach country, league, and team names for readable analysis. The joins do not change the meaning of a row: one row is still one match.

In [6]:
match_base = match_selected.copy()

match_base["date"] = pd.to_datetime(match_base["date"])

print(match_base.shape)
match_base.head()

def create_result(row):
    if row["home_team_goal"] > row["away_team_goal"]:
        return "Home Win"
    elif row["home_team_goal"] < row["away_team_goal"]:
        return "Away Win"
    else:
        return "Draw"


match_base["target"] = match_base.apply(
    create_result,
    axis=1
)

country_lookup = country[
    ["id", "name"]
].rename(
    columns={
        "id": "country_id",
        "name": "country_name"
    }
)

match_base = match_base.merge(
    country_lookup,
    on="country_id",
    how="left"
)

league_lookup = league[
    ["id", "name"]
].rename(
    columns={
        "id": "league_id",
        "name": "league_name"
    }
)

match_base = match_base.merge(
    league_lookup,
    on="league_id",
    how="left"
)

home_team_lookup = team_selected.rename(
    columns={
        "team_api_id": "home_team_api_id",
        "team_long_name": "home_team_name"
    }
)

match_base = match_base.merge(
    home_team_lookup,
    on="home_team_api_id",
    how="left"
)

away_team_lookup = team_selected.rename(
    columns={
        "team_api_id": "away_team_api_id",
        "team_long_name": "away_team_name"
    }
)

match_base = match_base.merge(
    away_team_lookup,
    on="away_team_api_id",
    how="left"
)

basic_columns = [
    "date",
    "season",
    "stage",
    "country_name",
    "league_name",
    "home_team_name",
    "away_team_name",
    "home_team_goal",
    "away_team_goal",
    "target"
]

match_base[basic_columns].head(10)

print("Rows:", len(match_base))
print("Duplicate rows:", match_base["id"].duplicated().sum())

display(match_base["target"].value_counts().rename_axis("Outcome").to_frame("Matches"))

(25979, 77)


Rows: 25979
Duplicate rows: 0


,Matches
Outcome,
Home Win,11917
Away Win,7466
Draw,6596


### Finding

The target has three classes and Home Win is the largest. Notebook 02 examines that imbalance in detail; later notebooks therefore use Macro F1 as the primary metric and accuracy as a secondary metric.

## 5. Add Historical Team Attributes

`merge_asof(..., direction="backward")` selects the most recent team snapshot whose date is no later than the match date. It is performed separately for home and away teams.

This is the core temporal safeguard: a future team record is never moved backward into an earlier match. The source has dates but no times, so an attribute record on the same calendar date is allowed by the preserved workflow. Switching to strictly earlier dates would change the existing dataset and is therefore not done during this readability cleanup.

In [7]:
# make sure dates are datetime
match_base["date"] = pd.to_datetime(match_base["date"])

team_attributes_selected["date"] = pd.to_datetime(
    team_attributes_selected["date"]
)

home_team_attributes = team_attributes_selected.copy()

home_team_attributes = home_team_attributes.rename(
    columns={
        "team_api_id": "home_team_api_id",
        "date": "home_team_attr_date"
    }
)

# add "home_" to all actual feature columns
home_team_attributes = home_team_attributes.rename(
    columns={
        col: f"home_{col}"
        for col in home_team_attributes.columns
        if col not in ["home_team_api_id", "home_team_attr_date"]
    }
)

match_base = match_base.sort_values(
    ["date", "home_team_api_id"]
).reset_index(drop=True)

home_team_attributes = home_team_attributes.sort_values(
    ["home_team_attr_date", "home_team_api_id"]
).reset_index(drop=True)

match_with_team_attrs = pd.merge_asof(
    match_base,
    home_team_attributes,

    left_on="date",
    right_on="home_team_attr_date",

    left_by="home_team_api_id",
    right_by="home_team_api_id",

    direction="backward"
)

(
    match_with_team_attrs["home_team_attr_date"]
    > match_with_team_attrs["date"]
).sum()

away_team_attributes = team_attributes_selected.copy()

away_team_attributes = away_team_attributes.rename(
    columns={
        "team_api_id": "away_team_api_id",
        "date": "away_team_attr_date"
    }
)

away_team_attributes = away_team_attributes.rename(
    columns={
        col: f"away_{col}"
        for col in away_team_attributes.columns
        if col not in ["away_team_api_id", "away_team_attr_date"]
    }
)

match_with_team_attrs = match_with_team_attrs.sort_values(
    ["date", "away_team_api_id"]
).reset_index(drop=True)

away_team_attributes = away_team_attributes.sort_values(
    ["away_team_attr_date", "away_team_api_id"]
).reset_index(drop=True)

match_with_team_attrs = pd.merge_asof(
    match_with_team_attrs,
    away_team_attributes,

    left_on="date",
    right_on="away_team_attr_date",

    left_by="away_team_api_id",
    right_by="away_team_api_id",

    direction="backward"
)

home_missing = match_with_team_attrs["home_buildUpPlaySpeed"].isnull()
away_missing = match_with_team_attrs["away_buildUpPlaySpeed"].isnull()

print(
    "Missing home team attributes:",
    match_with_team_attrs["home_buildUpPlaySpeed"].isnull().sum()
)

print(
    "Missing away team attributes:",
    match_with_team_attrs["away_buildUpPlaySpeed"].isnull().sum()
)

print("Rows:", len(match_with_team_attrs))
print(
    "Duplicate match IDs:",
    match_with_team_attrs["id"].duplicated().sum()
)

print("Match date range:")
print(match_base["date"].min(), "to", match_base["date"].max())

print("\nTeam attributes date range:")
print(
    team_attributes_selected["date"].min(),
    "to",
    team_attributes_selected["date"].max()
)

missing_by_season = (
    match_with_team_attrs
    .assign(
        missing_team_attributes=home_missing | away_missing
    )
    .groupby("season")["missing_team_attributes"]
    .agg(["sum", "count"])
)

missing_by_season["percentage"] = (
    missing_by_season["sum"]
    / missing_by_season["count"]
    * 100
)

missing_by_season

Missing home team attributes: 6063
Missing away team attributes: 6055
Rows: 25979
Duplicate match IDs: 0
Match date range:
2008-07-18 00:00:00 to 2016-05-25 00:00:00

Team attributes date range:
2010-02-22 00:00:00 to 2015-09-10 00:00:00


,sum,count,percentage
season,,,
2008/2009,3326,3326,100.000000
2009/2010,2143,3230,66.346749
2010/2011,211,3260,6.472393
2011/2012,143,3220,4.440994
2012/2013,225,3260,6.901840
2013/2014,189,3032,6.233509
2014/2015,231,3325,6.947368
2015/2016,156,3326,4.690319


### 5.1 Choose seasons with historical coverage

Team attribute history begins in February 2010, so the first two match seasons have large coverage gaps. The existing project decision keeps 2010/11 through 2015/16 and removes rows where either team lacks a historical tactical snapshot.

This is a coverage decision, not a random sample. It also establishes the six seasons later used for chronological train, validation, and test splits.

In [8]:
model_data = match_with_team_attrs[
    match_with_team_attrs["season"].isin([
        "2010/2011",
        "2011/2012",
        "2012/2013",
        "2013/2014",
        "2014/2015",
        "2015/2016"
    ])
].copy()

selected_season_match_count = len(model_data)
print("Matches kept:", selected_season_match_count)

team_attribute_check_columns = [
    "home_buildUpPlaySpeed",
    "away_buildUpPlaySpeed"
]

model_data = model_data.dropna(
    subset=team_attribute_check_columns
).copy()

team_attribute_match_count = len(model_data)
print("Matches remaining:", team_attribute_match_count)

print(
    "Missing home:",
    model_data["home_buildUpPlaySpeed"].isnull().sum()
)

print(
    "Missing away:",
    model_data["away_buildUpPlaySpeed"].isnull().sum()
)

print(
    "Duplicate matches:",
    model_data["id"].duplicated().sum()
)

display(model_data["target"].value_counts(normalize=True).mul(100).round(2).rename("Percent"))

Matches kept: 19423
Matches remaining: 18268
Missing home: 0
Missing away: 0
Duplicate matches: 0


target
Home Win    45.54
Away Win    29.09
Draw        25.37
Name: Percent, dtype: float64

## 6. Add Recent Form from Previous Matches

Recent form describes each team's previous five matches:

- total points (`3` for a win, `1` for a draw, `0` for a loss),
- average goals scored, and
- average goals conceded.

The crucial `shift(1)` moves the rolling window back one match. Without it, the current match result would leak into its own predictors. All available earlier seasons are retained as history before the modelling-season filter is applied.

In [9]:
# We calculate form from ALL historical matches
history_source = match_base[
    [
        "id",
        "date",
        "home_team_api_id",
        "away_team_api_id",
        "home_team_goal",
        "away_team_goal"
    ]
].copy()

history_source["date"] = pd.to_datetime(history_source["date"])

home_history = history_source[
    [
        "id",
        "date",
        "home_team_api_id",
        "away_team_api_id",
        "home_team_goal",
        "away_team_goal"
    ]
].copy()

home_history.columns = [
    "match_id",
    "date",
    "team_api_id",
    "opponent_api_id",
    "goals_for",
    "goals_against"
]

away_history = history_source[
    [
        "id",
        "date",
        "away_team_api_id",
        "home_team_api_id",
        "away_team_goal",
        "home_team_goal"
    ]
].copy()

away_history.columns = [
    "match_id",
    "date",
    "team_api_id",
    "opponent_api_id",
    "goals_for",
    "goals_against"
]

team_history = pd.concat(
    [home_history, away_history],
    ignore_index=True
)

team_history.head()

def calculate_points(row):
    if row["goals_for"] > row["goals_against"]:
        return 3
    elif row["goals_for"] == row["goals_against"]:
        return 1
    else:
        return 0


team_history["points"] = team_history.apply(
    calculate_points,
    axis=1
)

team_history = team_history.sort_values(
    ["team_api_id", "date", "match_id"]
).reset_index(drop=True)

team_history["form_points_5"] = (
    team_history
    .groupby("team_api_id")["points"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=5,
            min_periods=5
        ).sum()
    )
)

team_history["avg_goals_scored_5"] = (
    team_history
    .groupby("team_api_id")["goals_for"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=5,
            min_periods=5
        ).mean()
    )
)

team_history["avg_goals_conceded_5"] = (
    team_history
    .groupby("team_api_id")["goals_against"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=5,
            min_periods=5
        ).mean()
    )
)

example_team = team_history["team_api_id"].iloc[0]

team_history[
    team_history["team_api_id"] == example_team
][
    [
        "date",
        "goals_for",
        "goals_against",
        "points",
        "form_points_5",
        "avg_goals_scored_5",
        "avg_goals_conceded_5"
    ]
].head(12)

,date,goals_for,goals_against,points,form_points_5,avg_goals_scored_5,avg_goals_conceded_5
0,2008-08-10,0,0,1,NaN,NaN,NaN
1,2008-08-16,2,1,3,NaN,NaN,NaN
2,2008-08-22,2,1,3,NaN,NaN,NaN
3,2008-08-30,0,3,0,NaN,NaN,NaN
4,2008-09-12,2,0,3,NaN,NaN,NaN
5,2008-09-20,0,3,0,10.0,1.2,1.0
6,2008-09-27,0,0,1,9.0,1.2,1.6
7,2008-10-04,0,1,0,7.0,0.8,1.4
8,2008-10-19,2,0,3,4.0,0.4,1.4
9,2008-10-24,0,0,1,7.0,0.8,0.8


### 6.1 Join home and away form

The team-level history is joined back twice: once for the home team and once for the away team. Teams without five earlier matches do not yet have a complete form window, so those matches are removed by the preserved workflow.

In [10]:
home_form = team_history[
    [
        "match_id",
        "team_api_id",
        "form_points_5",
        "avg_goals_scored_5",
        "avg_goals_conceded_5"
    ]
].copy()

home_form = home_form.rename(
    columns={
        "team_api_id": "home_team_api_id",
        "form_points_5": "home_form_points_5",
        "avg_goals_scored_5": "home_avg_goals_scored_5",
        "avg_goals_conceded_5": "home_avg_goals_conceded_5"
    }
)

model_data = model_data.merge(
    home_form,
    left_on=["id", "home_team_api_id"],
    right_on=["match_id", "home_team_api_id"],
    how="left"
)

model_data.drop(
    columns=["match_id"],
    inplace=True
)

away_form = team_history[
    [
        "match_id",
        "team_api_id",
        "form_points_5",
        "avg_goals_scored_5",
        "avg_goals_conceded_5"
    ]
].copy()

away_form = away_form.rename(
    columns={
        "team_api_id": "away_team_api_id",
        "form_points_5": "away_form_points_5",
        "avg_goals_scored_5": "away_avg_goals_scored_5",
        "avg_goals_conceded_5": "away_avg_goals_conceded_5"
    }
)

model_data = model_data.merge(
    away_form,
    left_on=["id", "away_team_api_id"],
    right_on=["match_id", "away_team_api_id"],
    how="left"
)

model_data.drop(
    columns=["match_id"],
    inplace=True
)

form_feature_columns = [
    "home_form_points_5",
    "away_form_points_5",
    "home_avg_goals_scored_5",
    "away_avg_goals_scored_5",
    "home_avg_goals_conceded_5",
    "away_avg_goals_conceded_5"
]
model_data[form_feature_columns].isnull().sum()

missing_form_rows = model_data[
    form_feature_columns
].isnull().any(axis=1).sum()

print("Matches without complete previous-5 form:", missing_form_rows)

print(
    "Percentage:",
    missing_form_rows / len(model_data) * 100
)

print("Rows:", len(model_data))
print(
    "Duplicate match IDs:",
    model_data["id"].duplicated().sum()
)

model_data = model_data.dropna(
    subset=form_feature_columns
).copy()

form_match_count = len(model_data)
print("Rows remaining:", form_match_count)
print("Duplicate match IDs:", model_data["id"].duplicated().sum())

display(model_data[form_feature_columns].head())

Matches without complete previous-5 form: 252
Percentage: 1.3794613531858988
Rows: 18268
Duplicate match IDs: 0
Rows remaining: 18016
Duplicate match IDs: 0


,home_form_points_5,away_form_points_5,home_avg_goals_scored_5,away_avg_goals_scored_5,home_avg_goals_conceded_5,away_avg_goals_conceded_5
0,9.0,4.0,2.4,1.8,0.6,2.6
1,7.0,6.0,1.0,1.6,1.6,1.8
2,13.0,3.0,2.8,1.0,0.4,2.4
3,12.0,4.0,2.4,1.2,0.8,2.6
4,9.0,15.0,1.8,3.0,1.4,0.4


### Finding

The form variables use only previous results. This turns raw match history into information that could realistically have been known before kickoff.

## 7. Add Historical Player Strength

### 7.1 Create interpretable player skill groups

Individual skill columns are averaged into attack, passing, technical, pace, physical, defence, and goalkeeper scores. Pandas ignores missing component values when taking a row mean, so a composite can occasionally be based on fewer component skills; this is part of the preserved engineering logic.

In [11]:
pa = player_attributes_selected.copy()

pa["attack_score"] = pa[
    [
        "finishing",
        "shot_power",
        "long_shots",
        "positioning"
    ]
].mean(axis=1)

pa["passing_score"] = pa[
    [
        "short_passing",
        "long_passing",
        "vision"
    ]
].mean(axis=1)

pa["technical_score"] = pa[
    [
        "dribbling",
        "ball_control"
    ]
].mean(axis=1)

pa["pace_score"] = pa[
    [
        "acceleration",
        "sprint_speed",
        "agility"
    ]
].mean(axis=1)

pa["physical_score"] = pa[
    [
        "stamina",
        "strength",
        "jumping",
        "aggression"
    ]
].mean(axis=1)

pa["defence_score"] = pa[
    [
        "interceptions",
        "marking",
        "standing_tackle",
        "sliding_tackle"
    ]
].mean(axis=1)

pa["goalkeeper_score"] = pa[
    [
        "gk_diving",
        "gk_handling",
        "gk_kicking",
        "gk_positioning",
        "gk_reflexes"
    ]
].mean(axis=1)

player_features = pa[
    [
        "player_api_id",
        "date",
        "overall_rating",
        "potential",
        "attack_score",
        "passing_score",
        "technical_score",
        "pace_score",
        "physical_score",
        "defence_score",
        "goalkeeper_score"
    ]
].copy()

player_features["date"] = pd.to_datetime(
    player_features["date"]
)

print("Historical player feature rows:", len(player_features))
display(player_features.head())

Historical player feature rows: 183978


,player_api_id,date,overall_rating,potential,attack_score,passing_score,technical_score,pace_score,physical_score,defence_score,goalkeeper_score
0,505942,2016-02-18,67.0,71.0,44.75,59.666667,50.0,61.0,64.75,68.25,8.6
1,505942,2015-11-19,67.0,71.0,44.75,59.666667,50.0,61.0,64.75,68.25,8.6
2,505942,2015-09-21,62.0,66.0,44.75,59.666667,50.0,61.0,62.75,60.25,8.6
3,505942,2015-03-20,61.0,65.0,43.75,58.666667,49.0,61.0,62.50,57.75,7.6
4,505942,2007-02-22,61.0,65.0,43.75,58.666667,49.0,61.0,62.50,57.75,7.6


### 7.2 Reshape lineups and make the historical join

The 22 lineup columns are reshaped to a long table so each row represents one player in one match. A backward date join then selects the latest player snapshot dated on or before the match date.

The source contains 836 extra player/date rows across 835 duplicated player/date groups. The original workflow keeps the last record, which is preserved here to avoid changing completed results. Inspection found that the last record can be blank in these groups, so this rule is an explicit data-quality caveat for future methodology review.

In [12]:
home_player_cols = [f"home_player_{i}" for i in range(1, 12)]
away_player_cols = [f"away_player_{i}" for i in range(1, 12)]

lineup_check = model_data[["id"]].copy()

lineup_check["home_players_available"] = (
    model_data[home_player_cols].notna().sum(axis=1)
)

lineup_check["away_players_available"] = (
    model_data[away_player_cols].notna().sum(axis=1)
)

lineup_check[
    ["home_players_available", "away_players_available"]
].describe()

print(
    "Full home lineups:",
    (lineup_check["home_players_available"] == 11).sum()
)

print(
    "Full away lineups:",
    (lineup_check["away_players_available"] == 11).sum()
)

home_players_long = model_data[
    ["id", "date"] + home_player_cols
].melt(
    id_vars=["id", "date"],
    value_vars=home_player_cols,
    var_name="player_slot",
    value_name="player_api_id"
)

home_players_long["side"] = "home"

away_players_long = model_data[
    ["id", "date"] + away_player_cols
].melt(
    id_vars=["id", "date"],
    value_vars=away_player_cols,
    var_name="player_slot",
    value_name="player_api_id"
)

away_players_long["side"] = "away"

players_long = pd.concat(
    [home_players_long, away_players_long],
    ignore_index=True
)

players_long = players_long.dropna(
    subset=["player_api_id"]
).copy()

players_long["player_api_id"] = (
    players_long["player_api_id"].astype("int64")
)

player_features_join = player_features.copy()

player_features_join = player_features_join.rename(
    columns={
        "date": "attribute_date"
    }
)

print(
    "Duplicate player/date rows:",
    player_features_join.duplicated(
        subset=["player_api_id", "attribute_date"]
    ).sum()
)

player_features_join = (
    player_features_join
    .drop_duplicates(
        subset=["player_api_id", "attribute_date"],
        keep="last"
    )
)

players_long = players_long.sort_values(
    ["date", "player_api_id"]
).reset_index(drop=True)

player_features_join = player_features_join.sort_values(
    ["attribute_date", "player_api_id"]
).reset_index(drop=True)

players_with_attributes = pd.merge_asof(
    players_long,
    player_features_join,

    left_on="date",
    right_on="attribute_date",

    by="player_api_id",

    direction="backward"
)

future_player_data = (
    players_with_attributes["attribute_date"]
    > players_with_attributes["date"]
).sum()

print("Future player records used:", future_player_data)

print(
    "Player rows:",
    len(players_with_attributes)
)

print(
    "Missing historical overall ratings:",
    players_with_attributes["overall_rating"]
    .isnull()
    .sum()
)

missing_player_rating_pct = (
    players_with_attributes["overall_rating"]
    .isnull()
    .mean()
    * 100
)

print(
    "Missing player attribute percentage:",
    missing_player_rating_pct
)

Full home lineups: 16741
Full away lineups: 16717
Duplicate player/date rows: 836


Future player records used: 0
Player rows: 390466
Missing historical overall ratings: 274
Missing player attribute percentage: 0.07017256304005984


### 7.3 Aggregate players into team-level features

For each match side, the player with the highest goalkeeper score is treated as the goalkeeper. The remaining players are aggregated as outfield players. The result contains one set of average strength features for the home lineup and one for the away lineup.

Matches are kept only when at least eight players on both sides have historical attributes. This avoids estimating team strength from a very small part of a lineup while retaining most matches.

In [13]:
players_with_attributes["is_goalkeeper"] = False

valid_gk = players_with_attributes.dropna(
    subset=["goalkeeper_score"]
)

gk_indices = (
    valid_gk
    .groupby(["id", "side"])["goalkeeper_score"]
    .idxmax()
)

players_with_attributes.loc[
    gk_indices,
    "is_goalkeeper"
] = True

lineup_summary = (
    players_with_attributes
    .groupby(["id", "side"])
    .agg(
        players_listed=("player_api_id", "count"),
        players_with_attributes=("overall_rating", "count"),
        avg_overall_rating=("overall_rating", "mean"),
        avg_potential=("potential", "mean"),
        goalkeeper_rating=("goalkeeper_score", "max")
    )
    .reset_index()
)

outfield_players = players_with_attributes[
    ~players_with_attributes["is_goalkeeper"]
].copy()

outfield_summary = (
    outfield_players
    .groupby(["id", "side"])
    .agg(
        avg_attack=("attack_score", "mean"),
        avg_passing=("passing_score", "mean"),
        avg_technical=("technical_score", "mean"),
        avg_pace=("pace_score", "mean"),
        avg_physical=("physical_score", "mean"),
        avg_defence=("defence_score", "mean")
    )
    .reset_index()
)

player_team_features = lineup_summary.merge(
    outfield_summary,
    on=["id", "side"],
    how="left"
)

player_team_features_wide = (
    player_team_features
    .pivot(
        index="id",
        columns="side"
    )
)

player_team_features_wide.columns = [
    f"{side}_{feature}"
    for feature, side in player_team_features_wide.columns
]

player_team_features_wide = (
    player_team_features_wide
    .reset_index()
)

model_data = model_data.merge(
    player_team_features_wide,
    on="id",
    how="left"
)

print("Rows:", len(model_data))
print(
    "Duplicate match IDs:",
    model_data["id"].duplicated().sum()
)

print(
    model_data[
        [
            "home_players_with_attributes",
            "away_players_with_attributes"
        ]
    ].describe()
)

print(
    "Missing home player rating:",
    model_data["home_avg_overall_rating"]
    .isnull()
    .sum()
)

print(
    "Missing away player rating:",
    model_data["away_avg_overall_rating"]
    .isnull()
    .sum()
)

print(
    "Matches with fewer than 8 rated home players:",
    (
        model_data["home_players_with_attributes"] < 8
    ).sum()
)

print(
    "Matches with fewer than 8 rated away players:",
    (
        model_data["away_players_with_attributes"] < 8
    ).sum()
)

low_player_coverage = (
    (model_data["home_players_with_attributes"] < 8)
    |
    (model_data["away_players_with_attributes"] < 8)
)

print("Matches with insufficient player coverage:", low_player_coverage.sum())

print(
    "Percentage:",
    low_player_coverage.mean() * 100
)

model_data = model_data[
    ~low_player_coverage
].copy()

final_match_count = len(model_data)
print("Rows remaining:", final_match_count)
print("Duplicate match IDs:", model_data["id"].duplicated().sum())

display(model_data[["home_players_with_attributes", "away_players_with_attributes"]].describe())

Rows: 18016
Duplicate match IDs: 0
       home_players_with_attributes  away_players_with_attributes
count                  18016.000000                  18016.000000
mean                      10.830373                     10.827709
std                        0.769731                      0.780885
min                        2.000000                      2.000000
25%                       11.000000                     11.000000
50%                       11.000000                     11.000000
75%                       11.000000                     11.000000
max                       11.000000                     11.000000
Missing home player rating: 0
Missing away player rating: 0
Matches with fewer than 8 rated home players: 274
Matches with fewer than 8 rated away players: 288
Matches with insufficient player coverage: 407
Percentage: 2.259103019538188
Rows remaining: 17609
Duplicate match IDs: 0


,home_players_with_attributes,away_players_with_attributes
count,17609.000000,17609.000000
mean,10.926288,10.926288
std,0.329381,0.328690
min,8.000000,8.000000
25%,11.000000,11.000000
50%,11.000000,11.000000
75%,11.000000,11.000000
max,11.000000,11.000000


## 8. Create Home-minus-Away Difference Features

Absolute home and away strengths remain available, but football outcomes often depend on the *relative* strength of the two sides. A positive difference means the home team has the larger value; a negative difference favours the away team.

The same simple subtraction is used for player strength, recent form, and team tactics.

In [14]:
model_data["overall_rating_diff"] = (
    model_data["home_avg_overall_rating"]
    - model_data["away_avg_overall_rating"]
)

model_data["potential_diff"] = (
    model_data["home_avg_potential"]
    - model_data["away_avg_potential"]
)

model_data["attack_diff"] = (
    model_data["home_avg_attack"]
    - model_data["away_avg_attack"]
)

model_data["passing_diff"] = (
    model_data["home_avg_passing"]
    - model_data["away_avg_passing"]
)

model_data["technical_diff"] = (
    model_data["home_avg_technical"]
    - model_data["away_avg_technical"]
)

model_data["pace_diff"] = (
    model_data["home_avg_pace"]
    - model_data["away_avg_pace"]
)

model_data["physical_diff"] = (
    model_data["home_avg_physical"]
    - model_data["away_avg_physical"]
)

model_data["defence_diff"] = (
    model_data["home_avg_defence"]
    - model_data["away_avg_defence"]
)

model_data["goalkeeper_diff"] = (
    model_data["home_goalkeeper_rating"]
    - model_data["away_goalkeeper_rating"]
)

model_data["form_points_diff"] = (
    model_data["home_form_points_5"]
    - model_data["away_form_points_5"]
)

model_data["recent_goals_scored_diff"] = (
    model_data["home_avg_goals_scored_5"]
    - model_data["away_avg_goals_scored_5"]
)

model_data["recent_goals_conceded_diff"] = (
    model_data["home_avg_goals_conceded_5"]
    - model_data["away_avg_goals_conceded_5"]
)

model_data["buildUpPlaySpeed_diff"] = (
    model_data["home_buildUpPlaySpeed"]
    - model_data["away_buildUpPlaySpeed"]
)

model_data["buildUpPlayPassing_diff"] = (
    model_data["home_buildUpPlayPassing"]
    - model_data["away_buildUpPlayPassing"]
)

model_data["chanceCreationPassing_diff"] = (
    model_data["home_chanceCreationPassing"]
    - model_data["away_chanceCreationPassing"]
)

model_data["chanceCreationCrossing_diff"] = (
    model_data["home_chanceCreationCrossing"]
    - model_data["away_chanceCreationCrossing"]
)

model_data["chanceCreationShooting_diff"] = (
    model_data["home_chanceCreationShooting"]
    - model_data["away_chanceCreationShooting"]
)

model_data["defencePressure_diff"] = (
    model_data["home_defencePressure"]
    - model_data["away_defencePressure"]
)

model_data["defenceAggression_diff"] = (
    model_data["home_defenceAggression"]
    - model_data["away_defenceAggression"]
)

model_data["defenceTeamWidth_diff"] = (
    model_data["home_defenceTeamWidth"]
    - model_data["away_defenceTeamWidth"]
)

difference_columns = [column for column in model_data.columns if column.endswith("_diff")]
display(model_data[["home_team_name", "away_team_name"] + difference_columns + ["target"]].head())

,home_team_name,away_team_name,overall_rating_diff,potential_diff,attack_diff,passing_diff,technical_diff,pace_diff,physical_diff,defence_diff,...,recent_goals_conceded_diff,buildUpPlaySpeed_diff,buildUpPlayPassing_diff,chanceCreationPassing_diff,chanceCreationCrossing_diff,chanceCreationShooting_diff,defencePressure_diff,defenceAggression_diff,defenceTeamWidth_diff,target
0,Grasshopper Club Zürich,Neuchâtel Xamax,-3.000000,-2.090909,-6.800,-3.400000,-2.10,-4.283333,-4.200,-2.241667,...,-2.0,-30.0,5.0,-5.0,0.0,-5.0,-10.0,15.0,5.0,Draw
1,AC Bellinzona,FC Sion,-3.000000,-5.909091,-5.200,-5.233333,-4.80,-4.433333,-3.525,-3.875000,...,-0.2,5.0,-10.0,5.0,5.0,-5.0,10.0,10.0,10.0,Away Win
2,FC Luzern,FC St. Gallen,2.090909,2.818182,4.150,4.466667,5.05,4.000000,3.100,-2.200000,...,-2.0,10.0,0.0,10.0,15.0,5.0,-10.0,-10.0,0.0,Home Win
3,FC Basel,FC Zürich,3.454545,3.727273,0.800,2.966667,4.65,4.100000,5.225,9.200000,...,-1.8,-5.0,-15.0,5.0,-5.0,0.0,10.0,5.0,10.0,Home Win
4,FC Sion,FC Basel,-4.727273,-6.090909,-2.525,-3.966667,-6.60,-7.133333,-3.650,-6.775000,...,1.0,-5.0,15.0,0.0,-5.0,10.0,-25.0,-20.0,-15.0,Away Win


## 9. Audit Legitimate Candidate Predictors

The engineered file deliberately contains 156 columns because it preserves useful identifiers and traceability fields alongside modelling fields. Only 75 are legitimate initial predictors.

Columns are removed from the candidate set for conceptual reasons, not because of statistical feature selection:

- **identifiers:** database keys identify a record but do not describe match strength;
- **metadata:** dates and readable names support auditing rather than prediction;
- **post-match leakage:** final goals determine the target and cannot be predictors;
- **raw player IDs:** high-cardinality identities are replaced by historical aggregates;
- **formation coordinates:** raw lineup coordinates are not part of the chosen methodology; and
- **attribute dates:** dates are used to enforce historical joins, not as strength measures.

Notebook 03 will perform preprocessing. Notebook 04 will perform statistical and model-based feature analysis.

In [15]:
identifier_columns = [
    "id",
    "match_api_id",
    "country_id",
    "league_id",
    "home_team_api_id",
    "away_team_api_id"
]

leakage_columns = [
    "home_team_goal",
    "away_team_goal"
]

metadata_columns = [
    "date",
    "country_name",
    "home_team_name",
    "away_team_name"
]

player_id_columns = [
    f"home_player_{i}" for i in range(1, 12)
] + [
    f"away_player_{i}" for i in range(1, 12)
]

player_position_columns = [
    col for col in model_data.columns
    if "player_X" in col or "player_Y" in col
]

attribute_date_columns = [
    "home_team_attr_date",
    "away_team_attr_date"
]

exclusion_groups = {
    "Identifiers": identifier_columns,
    "Metadata": metadata_columns,
    "Post-match / leakage": leakage_columns,
    "Raw player IDs": player_id_columns,
    "Raw formation coordinates": player_position_columns,
    "Historical join dates": attribute_date_columns,
}

columns_to_exclude = [
    column
    for group in exclusion_groups.values()
    for column in group
    if column in model_data.columns
]

X_candidate = model_data.drop(columns=columns_to_exclude + ["target"]).copy()
y = model_data["target"].copy()

candidate_categorical = X_candidate.select_dtypes(exclude="number").columns.tolist()
candidate_numerical = X_candidate.select_dtypes(include="number").columns.tolist()

exclusion_summary = pd.DataFrame(
    {
        "Reason": exclusion_groups.keys(),
        "Columns removed": [
            sum(column in model_data.columns for column in columns)
            for columns in exclusion_groups.values()
        ],
    }
)

display(exclusion_summary)
print("Full engineered columns:", model_data.shape[1])
print("Candidate features:", X_candidate.shape[1])
print("Numerical candidates:", len(candidate_numerical))
print("Categorical candidates:", len(candidate_categorical))
print("Categorical feature names:", candidate_categorical)

,Reason,Columns removed
0,Identifiers,6
1,Metadata,4
2,Post-match / leakage,2
3,Raw player IDs,22
4,Raw formation coordinates,44
5,Historical join dates,2


Full engineered columns: 156
Candidate features: 75
Numerical candidates: 65
Categorical candidates: 10
Categorical feature names: ['season', 'league_name', 'home_buildUpPlayDribblingClass', 'home_buildUpPlayPositioningClass', 'home_chanceCreationPositioningClass', 'home_defenceDefenderLineClass', 'away_buildUpPlayDribblingClass', 'away_buildUpPlayPositioningClass', 'away_chanceCreationPositioningClass', 'away_defenceDefenderLineClass']


### 9.1 Final integrity checks

We expect one row per match, no duplicate match IDs, and no missing or infinite values among the 75 candidates. Missing values can still exist in excluded raw traceability columns; the zero-missing claim applies only to the candidate matrix.

In [16]:
numeric_candidates = X_candidate.select_dtypes(include="number")

integrity_checks = pd.Series(
    {
        "Engineered matches": len(model_data),
        "Full engineered columns": model_data.shape[1],
        "Candidate features": X_candidate.shape[1],
        "Duplicate match IDs": int(model_data["id"].duplicated().sum()),
        "Missing candidate values": int(X_candidate.isna().sum().sum()),
        "Infinite candidate values": int(
            numeric_candidates.isin([float("inf"), float("-inf")]).sum().sum()
        ),
    },
    name="Value",
)

display(integrity_checks.to_frame())
display(model_data["target"].value_counts().rename_axis("Outcome").to_frame("Matches"))

assert model_data.shape == (17609, 156)
assert X_candidate.shape == (17609, 75)
assert model_data["id"].duplicated().sum() == 0
assert X_candidate.isna().sum().sum() == 0

,Value
Engineered matches,17609
Full engineered columns,156
Candidate features,75
Duplicate match IDs,0
Missing candidate values,0
Infinite candidate values,0


,Matches
Outcome,
Home Win,8026
Away Win,5127
Draw,4456


### Finding

The completed workflow produces **17,609 matches**, **156 total engineered columns**, and **75 legitimate candidate predictors before one-hot encoding**. Candidate predictors have no missing values, and match IDs are unique.

The row flow is:

```text
25,979 source matches
        ↓ keep 2010/11–2015/16
19,423 matches
        ↓ require historical team attributes
18,268 matches
        ↓ require five previous matches for form
18,016 matches
        ↓ require at least eight rated players per side
17,609 engineered matches
```

## 10. Save the Engineered Dataset and Summary

To respect the project's source-data protection rule, this cleanup does not overwrite an existing engineered CSV. If the file is absent, the notebook creates it; if it already exists, the in-memory reconstruction is checked and the existing artifact is preserved.

The compact summary table is a generated report and may be refreshed safely.

In [17]:
engineered_path = PROCESSED_DIR / "model_data_engineered.csv"
summary_path = TABLES_DIR / "dataset_summary.csv"

if engineered_path.exists():
    print(f"Protected existing engineered dataset: {engineered_path}")
else:
    model_data.to_csv(engineered_path, index=False)
    print(f"Saved engineered dataset: {engineered_path}")

dataset_summary = pd.DataFrame(
    {
        "metric": [
            "Number of matches",
            "Candidate ML features",
            "Duplicate match IDs",
            "Missing candidate values",
            "Home Win %",
            "Away Win %",
            "Draw %",
        ],
        "value": [
            len(model_data),
            X_candidate.shape[1],
            model_data["id"].duplicated().sum(),
            X_candidate.isna().sum().sum(),
            model_data["target"].eq("Home Win").mean() * 100,
            model_data["target"].eq("Away Win").mean() * 100,
            model_data["target"].eq("Draw").mean() * 100,
        ],
    }
)

dataset_summary.to_csv(summary_path, index=False)
display(dataset_summary)

retention_table = pd.DataFrame(
    {
        "stage": [
            "Raw Match table",
            "Keep 2010/11–2015/16",
            "Require historical team attributes",
            "Require five previous matches for form",
            "Require at least eight rated players per side",
        ],
        "matches_retained": [
            raw_match_count,
            selected_season_match_count,
            team_attribute_match_count,
            form_match_count,
            final_match_count,
        ],
    }
)
retention_table["matches_removed_at_stage"] = (
    retention_table["matches_retained"].shift(1) - retention_table["matches_retained"]
).fillna(0).astype(int)
retention_table["percent_of_raw_retained"] = (
    retention_table["matches_retained"] / raw_match_count * 100
)
retention_table.to_csv(TABLES_DIR / "data_retention.csv", index=False)
display(retention_table.round(2))


Protected existing engineered dataset: data/processed/model_data_engineered.csv


,metric,value
0,Number of matches,17609.000000
1,Candidate ML features,75.000000
2,Duplicate match IDs,0.000000
3,Missing candidate values,0.000000
4,Home Win %,45.578965
5,Away Win %,29.115793
6,Draw %,25.305242


,stage,matches_retained,matches_removed_at_stage,percent_of_raw_retained
0,Raw Match table,25979,0,100.00
1,Keep 2010/11–2015/16,19423,6556,74.76
2,Require historical team attributes,18268,1155,70.32
3,Require five previous matches for form,18016,252,69.35
4,Require at least eight rated players per side,17609,407,67.78


# Notebook Findings and Handoff

- The relational source was converted from separate match, lookup, team-history, and player-history tables into one row per match.
- Backward time joins and shifted rolling windows prevent records dated after a match and the current match result from entering its predictors.
- Recent form, lineup strength, tactics, and home-minus-away differences describe information available at prediction time.
- The final dataset contains 17,609 matches and 75 valid initial candidate features, with no missing candidate values or duplicate match IDs.
- Same-calendar-date snapshots and the preserved player/date duplicate rule are documented limitations; changing either would be a methodology change rather than a readability edit.

**Next:** Notebook 02 explores patterns and data quality in the original tables. Notebook 03 then removes conceptually invalid columns, applies the chronological split, and one-hot encodes the valid categorical predictors.